# Challenge 2 — Steps 5–6: final model and predictions

**Team 5 | Kevin Herrera**

This notebook selects a final classifier using **training-only** stratified 5-fold cross-validation, fits it on all 9,247 training examples, and exports `predictions.csv` for the 3,083 unlabeled test examples. Run from the `Challenge 2/` directory after installing `requirements.txt`. No test labels are used or inferred.

## Step 5 — Model selection

Earlier teammates' reported tuned purchase-class F1 values (not independently revalidated here): **Random Forest 0.682 ± 0.008**, Logistic Regression **0.680 ± 0.009**, SVM **0.677 ± 0.010**, KNN **0.633 ± 0.022**, Naive Bayes **0.600 ± 0.011**. Tree-model baseline results were Decision Tree **0.563**, Random Forest **0.617**, Gradient Boosting **0.661**. These figures are **not directly comparable** until all models use identical preprocessing, folds and evaluation; Joaquin's PR #8 used additional log1p transformations. The 0.002 difference between the two top tuned models is smaller than the reported fold variation.

**Provisional selection:** tuned Random Forest, because its reported F1 is highest and its regularization reduces the baseline forest's severe overfitting (baseline train F1=1.000 vs CV F1=0.617). It is more computationally expensive and less interpretable than Logistic Regression; choose Logistic Regression instead if consistent re-evaluation demonstrates an equal or higher F1, or if simpler deployment is prioritized. The code below independently validates the selected Random Forest using the team's merged `helpers.py` and reports actual CV metrics before fitting.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report
from helpers import CV, SCORING, RANDOM_STATE, load_data, load_test, make_model_pipeline

X, y = load_data()
assert len(X) == 9247 and len(y) == 9247
assert 'ID' not in X.columns and 'Revenue' not in X.columns
print('scikit-learn:', sklearn.__version__)
print('Training data:', X.shape, 'buyers:', int(y.sum()))

### Selected model configuration

The Random Forest parameters below **match Jiayang's reported best GridSearchCV parameters** from PR #9 (`class_weight='balanced'`, `max_depth=None`, `min_samples_leaf=5`). The remaining parameters match his default RandomForestClassifier (`n_estimators=100`, `random_state=42`). The resulting CV F1 printed below is the value to report for this exact pipeline.

In [ ]:
final_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
pipeline = make_model_pipeline(final_model, scale=False)
cv_result = cross_validate(pipeline, X, y, cv=CV, scoring=SCORING, return_train_score=True, n_jobs=1)
metrics = {k: float(np.mean(v)) for k, v in cv_result.items() if k.startswith('test_') or k.startswith('train_')}
print('CV F1:', round(metrics['test_f1'], 4), '+/-', round(float(np.std(cv_result['test_f1'])), 4))
print('Train F1:', round(metrics['train_f1'], 4))
print('CV precision:', round(metrics['test_precision'], 4), 'recall:', round(metrics['test_recall'], 4), 'accuracy:', round(metrics['test_accuracy'], 4))
print('Train-CV F1 gap:', round(metrics['train_f1'] - metrics['test_f1'], 4))
cv_predictions = cross_val_predict(pipeline, X, y, cv=CV, n_jobs=1)
print('Out-of-fold confusion matrix:')
print(confusion_matrix(y, cv_predictions))
print(classification_report(y, cv_predictions, zero_division=0))

## Step 6 — Refit and export

The test data is loaded **only after model selection and cross-validation**. The `ID` column is retained for the submission, never used as a feature. Output is written beside this notebook as `Challenge 2/predictions.csv`.

In [ ]:
X_test, test_ids = load_test()
assert len(X_test) == 3083 and len(test_ids) == 3083
assert X_test.columns.tolist() == X.columns.tolist()
assert test_ids.is_unique and not test_ids.isna().any()
pipeline.fit(X, y)
predicted = pipeline.predict(X_test).astype(bool)
submission = pd.DataFrame({'ID': test_ids, 'Revenue': predicted})
assert submission.columns.tolist() == ['ID', 'Revenue']
assert len(submission) == 3083 and submission['Revenue'].notna().all()
assert submission['Revenue'].isin([True, False]).all()
output_path = Path('predictions.csv')
submission.to_csv(output_path, index=False)
check = pd.read_csv(output_path)
assert len(check) == 3083 and check.columns.tolist() == ['ID', 'Revenue']
assert check['ID'].equals(test_ids.reset_index(drop=True))
print('Saved:', output_path.resolve())
print('Rows:', len(check), '| predicted purchases:', int(predicted.sum()))
print(check.head())

## Reproducibility and integration checks

1. From `Challenge 2/`, install `pip install -r requirements.txt`, then **Restart & Run All**.
2. Confirm the printed CV F1; do not copy the provisional 0.682 from a teammate's different model configuration as this notebook's measured score.
3. Verify that Random Forest CV F1 reproduces Jiayang's reported 0.682 ± 0.008 from PR #9; re-evaluate Logistic Regression using the same preprocessing before final sign-off.
4. Commit the generated `predictions.csv` and verify exactly 3,083 rows, `ID,Revenue`, boolean predictions and original test ID order.
5. After PRs #8 and #9 are integrated, update README with actual measured final score and rerun this notebook. All team members must complete their own Canvas peer-review form.